In [ ]:
import os 
import wandb
import torch

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize
from pytorch_lightning.callbacks import ModelCheckpoint

from codefiles.helpers import set_all_seeds, build_model, build_lightningmodule, build_datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

def main(cfg) -> None:
    wandb.finish()
    set_all_seeds(seed=cfg.seed)
    wandb.init(
        project=cfg.wandb.project,
        group=None if cfg.wandb.group == "None" else cfg.wandb.group,
        config={key: value for key, value in cfg.items()},
    )
    
    checkpointaddon = ""
    if "corrupted_data_protocol" in cfg.modelname:
        if cfg.modelname.corrupted_data_protocol:
            checkpointaddon = "_corrupted"
        else:
            checkpointaddon = "_clean"

    checkpoint_name = f"debugmodel{checkpointaddon}"
    if os.path.exists(f'/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints/{checkpoint_name}.ckpt'):
        os.remove(f'/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints/{checkpoint_name}.ckpt')
    
    checkpoint_callback = ModelCheckpoint(
        monitor=cfg.encoders.monitor.metric, mode=cfg.encoders.monitor.mode,
        dirpath=f"/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints",
        filename=checkpoint_name,
        save_top_k=1,
    )

    model = build_model(cfg)
    lightningmodule = build_lightningmodule(cfg, model)
    datamodule = build_datamodule(cfg)

    trainer = pl.Trainer(
        logger=WandbLogger(project=cfg.wandb.project, dir="wandb/"),
        log_every_n_steps=1,
        accelerator='gpu',
        devices=1,
        max_epochs=cfg.max_epochs,
        precision=cfg.precision,
        enable_checkpointing=True,
        callbacks=[checkpoint_callback] 
    )

    trainer.fit(lightningmodule, datamodule)
    trainer.test(ckpt_path='best', datamodule=datamodule)

    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")
    main(cfg)

Global seed set to 420
/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:269: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/loggers/wandb.py:395: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.


/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:165: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/l ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:101: attribute 'model' removed from hparams because it cannot be pickled


total_samples: 4503 / 4503
no_missing: 4503 / 4503
1_missing: 0 / 4503
modality_0_missing: 0 / 4503
modality_1_missing: 0 / 4503
total_samples: 1476 / 1476
no_missing: 1476 / 1476
1_missing: 0 / 1476
modality_0_missing: 0 / 1476
modality_1_missing: 0 / 1476
total_samples: 1463 / 1463
no_missing: 1463 / 1463
1_missing: 0 / 1463
modality_0_missing: 0 / 1463
modality_1_missing: 0 / 1463


/sc-projects/sc-proj-ukb-cvd/environments/mml_rocm/lib/python3.9/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:613: Checkpoint directory /sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                    | Params
------------------------------------------------------
0 | model     | Multimodal_Architecture | 44.8 M
1 | loss      | CrossEntropyLoss        | 0     
2 | acc_train | MulticlassAccuracy      | 0     
3 | acc_val   | MulticlassAccuracy      | 0     
4 | acc_test  | MulticlassAccuracy      | 0     
------------------------------------------------------
44.8 M    Trainable params
0         Non-trainable params
44.8 M    Total params
179.046   Total estimated model params size (MB)
/sc-projects/sc-proj-ukb-cvd/projects/simple_mml_baseline_tr/codefiles/datasets/crema_d.py:75: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, 

Model GFlops (per instance): 27.24


...

Validation: 0it [00:00, ?it/s]

Added layer to modality 0.


Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]